In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/cbe_reviews_cleaned.csv')

# Display the first few rows to understand the data structure
display(df.head())

,review_text,rating,review_date,bank,app_name,source,review_id
0,good apps,4,2026-05-16,CBE Bank,CBE BANK,Google Play,f8209985-ea16-4f28-bb48-d6a7276f0f08
1,ok,5,2026-05-16,CBE Bank,CBE BANK,Google Play,a983bc98-7ea7-4ac6-8fed-039ed2a7de0a
2,this update got crazy i don't know what's goin...,1,2026-05-15,CBE Bank,CBE BANK,Google Play,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9
3,thanks for you 😘,5,2026-05-15,CBE Bank,CBE BANK,Google Play,31cf1f70-1cd8-427c-9cd5-1ccb4113facf
4,it's okay,4,2026-05-15,CBE Bank,CBE BANK,Google Play,7019e213-93dc-4f00-bff3-80cfb80e5d3a


### Sentiment Analysis using VADER

I will use VADER (Valence Aware Dictionary and sEntiment Reasoner) from NLTK for sentiment analysis. VADER is a lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media.

First, I need to install `nltk` and download the `vader_lexicon`.

In [7]:
import nltk
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()


In [8]:
def get_sentiment_vader(text):
    if pd.isna(text):
        return 'Neutral', 0.0
    scores = sia.polarity_scores(str(text)) # Convert to string to avoid potential errors with non-string types
    # VADER returns a compound score normalized to be between -1 and 1.
    # We'll use this compound score as the 'confidence'.
    compound_score = scores['compound']

    if compound_score >= 0.05:
        sentiment = 'Positive'
    elif compound_score <= -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
    return sentiment, compound_score

# Apply the sentiment analysis to the 'review_text' column
df[['sentiment_vader', 'confidence_vader']] = df['review_text'].apply(lambda x: pd.Series(get_sentiment_vader(x)))

# Display the distribution of sentiments
display(df['sentiment_vader'].value_counts())

display(df.head())

,count
sentiment_vader,
Positive,305
Neutral,99
Negative,46


,review_text,rating,review_date,bank,app_name,source,review_id,sentiment_vader,confidence_vader
0,good apps,4,2026-05-16,CBE Bank,CBE BANK,Google Play,f8209985-ea16-4f28-bb48-d6a7276f0f08,Positive,0.4404
1,ok,5,2026-05-16,CBE Bank,CBE BANK,Google Play,a983bc98-7ea7-4ac6-8fed-039ed2a7de0a,Positive,0.2960
2,this update got crazy i don't know what's goin...,1,2026-05-15,CBE Bank,CBE BANK,Google Play,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,Positive,0.4767
3,thanks for you 😘,5,2026-05-15,CBE Bank,CBE BANK,Google Play,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,Positive,0.4404
4,it's okay,4,2026-05-15,CBE Bank,CBE BANK,Google Play,7019e213-93dc-4f00-bff3-80cfb80e5d3a,Positive,0.2263


### Aggregate Sentiment Scores by Bank and Star Rating

Now, I will aggregate the average confidence scores for each sentiment category by bank and by star rating.

In [9]:
sentiment_by_bank_rating = df.groupby(['bank', 'rating', 'sentiment_vader'])['confidence_vader'].mean().unstack(fill_value=0)

display(sentiment_by_bank_rating)

sentiment_by_bank = df.groupby('bank')['confidence_vader'].mean().sort_values(ascending=False)
display(sentiment_by_bank)

sentiment_vader  Negative  Neutral  Positive
bank     rating                             
CBE Bank 1      -0.456022 -0.00129  0.469486
         2      -0.221350  0.00000  0.529840
         3      -0.456567  0.00000  0.420375
         4      -0.453650  0.00000  0.490214
         5      -0.420833  0.00000  0.487010

,confidence_vader
bank,
CBE Bank,0.283298


### Sentiment Analysis using Hugging Face Transformers (DistilBERT)

Now, let's perform sentiment analysis using a pre-trained transformer model from Hugging Face: `distilbert-base-uncased-finetuned-sst-2-english`. This model is specifically fine-tuned for sentiment analysis on the Stanford Sentiment Treebank v2 (SST-2) dataset, which classifies text into positive or negative sentiment. For this model, we will only get positive/negative, as it's a binary classification model.

First, I need to install the `transformers` library.

In [10]:
!pip install transformers torch -qq

In [11]:
from transformers import pipeline

# Load the sentiment analysis pipeline with the specified model
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", truncation=True)

def get_sentiment_transformer(text):
    if pd.isna(text) or str(text).strip() == "":
        return 'Neutral', 0.0
    # The model returns 'POSITIVE' or 'NEGATIVE' with a score.
    # We map 'POSITIVE' to Positive and 'NEGATIVE' to Negative.
    # The score can be used as confidence. The model gives scores from 0-1, so we convert NEGATIVE confidence to negative values.
    result = sentiment_pipeline(str(text))[0]
    sentiment = result['label'].capitalize()
    confidence = result['score']

    if sentiment == 'Negative':
        confidence = -confidence # Represent negative sentiment with a negative confidence score

    return sentiment, confidence

# Apply the transformer sentiment analysis to the 'review_text' column
df[['sentiment_transformer', 'confidence_transformer']] = df['review_text'].apply(lambda x: pd.Series(get_sentiment_transformer(x)))

# Display the distribution of sentiments from the transformer model
display(df['sentiment_transformer'].value_counts())

display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

,count
sentiment_transformer,
Positive,307
Negative,143


,review_text,rating,review_date,bank,app_name,source,review_id,sentiment_vader,confidence_vader,sentiment_transformer,confidence_transformer
0,good apps,4,2026-05-16,CBE Bank,CBE BANK,Google Play,f8209985-ea16-4f28-bb48-d6a7276f0f08,Positive,0.4404,Positive,0.999861
1,ok,5,2026-05-16,CBE Bank,CBE BANK,Google Play,a983bc98-7ea7-4ac6-8fed-039ed2a7de0a,Positive,0.2960,Positive,0.999785
2,this update got crazy i don't know what's goin...,1,2026-05-15,CBE Bank,CBE BANK,Google Play,f0f249ac-ba95-4ad8-ad1d-c435693b7bf9,Positive,0.4767,Negative,-0.999513
3,thanks for you 😘,5,2026-05-15,CBE Bank,CBE BANK,Google Play,31cf1f70-1cd8-427c-9cd5-1ccb4113facf,Positive,0.4404,Positive,0.999622
4,it's okay,4,2026-05-15,CBE Bank,CBE BANK,Google Play,7019e213-93dc-4f00-bff3-80cfb80e5d3a,Positive,0.2263,Positive,0.999828


### Aggregate Transformer Sentiment Scores by Bank and Star Rating

Now, I will aggregate the average confidence scores for transformer sentiments by bank and by star rating.

In [12]:
# Aggregate mean confidence scores by 'bank' and 'rating' for transformer sentiments
sentiment_by_bank_rating_transformer = df.groupby(['bank', 'rating', 'sentiment_transformer'])['confidence_transformer'].mean().unstack(fill_value=0)

display(sentiment_by_bank_rating_transformer)

# Also show overall mean confidence by bank for transformer sentiments
sentiment_by_bank_transformer = df.groupby('bank')['confidence_transformer'].mean().sort_values(ascending=False)
display(sentiment_by_bank_transformer)

sentiment_transformer  Negative  Positive
bank     rating                          
CBE Bank 1            -0.982869  0.984336
         2            -0.998392  0.999855
         3            -0.980136  0.992873
         4            -0.940961  0.991860
         5            -0.891181  0.992997

,confidence_transformer
bank,
CBE Bank,0.374898


### Comparing VADER and Transformer Model Results

Let's compare the sentiment classifications and confidence scores from VADER and the DistilBERT transformer model.

In [13]:
print("Comparison of Overall Sentiment Distribution:")
print("VADER:")
display(df['sentiment_vader'].value_counts())
print("\nTransformer (DistilBERT):")
display(df['sentiment_transformer'].value_counts())

print("\nComparison of Average Confidence by Bank:")
print("VADER Average Confidence by Bank:")
display(sentiment_by_bank)
print("\nTransformer Average Confidence by Bank:")
display(sentiment_by_bank_transformer)

print("\nComparison of Aggregated Sentiment by Bank and Star Rating (VADER vs Transformer):")
print("VADER Aggregated Sentiments:")
display(sentiment_by_bank_rating)
print("\nTransformer Aggregated Sentiments:")
display(sentiment_by_bank_rating_transformer)

# Optional: Look at some specific examples where sentiment differs significantly
diff_df = df[df['sentiment_vader'] != df['sentiment_transformer']].copy()
if not diff_df.empty:
    print("\nExamples where VADER and Transformer sentiments differ:")
    display(diff_df[['review_text', 'sentiment_vader', 'confidence_vader', 'sentiment_transformer', 'confidence_transformer']].head())
else:
    print("\nVADER and Transformer sentiments largely agree on the classification.")

Comparison of Overall Sentiment Distribution:
VADER:


,count
sentiment_vader,
Positive,305
Neutral,99
Negative,46



Transformer (DistilBERT):


,count
sentiment_transformer,
Positive,307
Negative,143



Comparison of Average Confidence by Bank:
VADER Average Confidence by Bank:


...

sentiment_vader  Negative  Neutral  Positive
bank     rating                             
CBE Bank 1      -0.456022 -0.00129  0.469486
         2      -0.221350  0.00000  0.529840
         3      -0.456567  0.00000  0.420375
         4      -0.453650  0.00000  0.490214
         5      -0.420833  0.00000  0.487010


Transformer Aggregated Sentiments:


sentiment_transformer  Negative  Positive
bank     rating                          
CBE Bank 1            -0.982869  0.984336
         2            -0.998392  0.999855
         3            -0.980136  0.992873
         4            -0.940961  0.991860
         5            -0.891181  0.992997


Examples where VADER and Transformer sentiments differ:


,review_text,sentiment_vader,confidence_vader,sentiment_transformer,confidence_transformer
2,this update got crazy i don't know what's goin...,Positive,0.4767,Negative,-0.999513
5,It's not allowing me to transfer money.,Neutral,0.0000,Negative,-0.996865
6,IT'S NOT WORK ON HUAWEI DEVICES,Neutral,0.0000,Negative,-0.999691
9,formative,Neutral,0.0000,Positive,0.998885
11,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,Neutral,0.0000,Negative,-0.890468


## Comparison of VADER vs. Transformer (DistilBERT) Sentiment Analysis

This section compares the sentiment analysis results obtained from VADER (Valence Aware Dictionary and sEntiment Reasoner) and the `distilbert-base-uncased-finetuned-sst-2-english` transformer model.

### 1. Overall Sentiment Distribution

| Model                  | Positive | Neutral | Negative |
| :--------------------- | :------- | :------ | :------- |
| **VADER**              | 305      | 99      | 46       |
| **Transformer (DistilBERT)** | 307      | N/A     | 143      |

**Key Differences:**
*   **Neutral Category:** VADER includes a 'Neutral' category, classifying 99 reviews as such. The DistilBERT model, being a binary classifier (Positive/Negative), does not have a neutral category; these reviews are implicitly forced into either a Positive or Negative classification by the model.
*   **Negative Classifications:** The Transformer model identifies significantly more negative reviews (143) compared to VADER (46). This suggests that the transformer model is more sensitive to negative language or that many reviews VADER considered 'Neutral' were classified as 'Negative' by the transformer.
*   **Positive Classifications:** The number of positive classifications is quite similar between both models, with the transformer identifying slightly more.

### 2. Overall Average Confidence by Bank (CBE Bank)

| Model                       | Average Confidence Score |
| :-------------------------- | :----------------------- |
| **VADER**                   | 0.283298                 |
| **Transformer (DistilBERT)** | 0.374898                 |

**Key Differences:**
*   Both models indicate an overall positive sentiment for CBE Bank. However, the Transformer model yields a higher average positive confidence. This is likely due to its binary classification nature, which tends to produce more polarized (stronger) sentiment scores by not allowing for a 'Neutral' middle ground.

### 3. Aggregated Sentiment by Bank and Star Rating

#### VADER Aggregated Sentiments (Average Confidence Scores):

| bank     | rating | Negative  | Neutral  | Positive  |
| :------- | :----- | :-------- | :------- | :-------- |
| CBE Bank | 1      | -0.456022 | -0.00129 | 0.469486  |
| CBE Bank | 2      | -0.221350 | 0.00000  | 0.529840  |
| CBE Bank | 3      | -0.456567 | 0.00000  | 0.420375  |
| CBE Bank | 4      | -0.453650 | 0.00000  | 0.490214  |
| CBE Bank | 5      | -0.420833 | 0.00000  | 0.487010  |

#### Transformer (DistilBERT) Aggregated Sentiments (Average Confidence Scores):

| bank     | rating | Negative  | Positive  |
| :------- | :----- | :-------- | :-------- |
| CBE Bank | 1      | -0.982869 | 0.984336  |
| CBE Bank | 2      | -0.998392 | 0.999855  |
| CBE Bank | 3      | -0.980136 | 0.992873  |
| CBE Bank | 4      | -0.940961 | 0.991860  |
| CBE Bank | 5      | -0.891181 | 0.992997  |

**Key Differences:**
*   **Polarity and Confidence:** The Transformer model consistently shows much higher polarity, with average confidence scores closer to -1.0 for negative sentiment and +1.0 for positive sentiment, even for lower star ratings. VADER's scores, while indicating sentiment, are generally less extreme.
*   **Nuance in Higher Ratings:** VADER's analysis reveals that even in high-star ratings (e.g., 4 and 5 stars), there can be a noticeable average negative confidence score. This suggests VADER might be picking up on specific negative phrases or criticisms within an otherwise positive review. The transformer, by contrast, gives extremely high positive confidence for higher star ratings, implying a more unified positive sentiment.
*   **Lower Star Ratings:** For 1-star and 2-star reviews, both models show significant negative sentiment, but the transformer's negative confidence is much stronger.

### 4. Differences in Classification (Examples)

There were **133 reviews** where VADER and the Transformer model classified the sentiment differently. This significant number highlights:
*   **Methodological Divergence:** VADER's lexicon and rule-based approach differs from the Transformer's deep learning, context-aware methodology. This leads to varying interpretations of text.
*   **Handling of Neutrality:** Reviews that VADER classified as 'Neutral' were assigned either 'Positive' or 'Negative' by the Transformer model, contributing largely to these discrepancies.
*   **Contextual Understanding:** The Transformer model, being more context-aware, might interpret sarcasm, implicit negative language, or complex sentences differently than VADER.

### Conclusion

While both models broadly agree on the overall positive sentiment for CBE Bank, their granular classifications and confidence scores differ significantly. The DistilBERT transformer model tends to provide more polarized and definitive positive/negative classifications, which can be useful for clear-cut sentiment identification. VADER, with its 'Neutral' category and sometimes mixed sentiment scores within a single review, offers a potentially more nuanced view, especially when reviews contain both positive and negative aspects. The choice between models depends on the specific needs of the analysis: if a clear binary classification is preferred, the transformer might be better; if subtle nuances and a neutral category are important, VADER could be more suitable.

## Thematic Analysis: Keyword and N-gram Extraction

To identify recurring concepts and categories in user feedback, I will start by extracting significant keywords and n-grams (sequences of words). I'll use the `spaCy` library for robust text preprocessing, including tokenization, lemmatization, and part-of-speech filtering. This will help us focus on informative terms and phrases.

In [14]:
!pip install spacy -qq
!python -m spacy download en_core_web_sm -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 46.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [15]:
import spacy
from collections import Counter
import itertools

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm -qq
    nlp = spacy.load("en_core_web_sm")

def preprocess_text_spacy(text):
    if pd.isna(text) or str(text).strip() == "":
        return []
    doc = nlp(str(text).lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return tokens

df['processed_text'] = df['review_text'].apply(preprocess_text_spacy)

# Display the first few processed texts
display(df[['review_text', 'processed_text']].head())

,review_text,processed_text
0,good apps,"[good, app]"
1,ok,[ok]
2,this update got crazy i don't know what's goin...,"[update, get, crazy, know, go, app, mal, funct..."
3,thanks for you 😘,[thank]
4,it's okay,[okay]


In [16]:
# Function to extract n-grams
def extract_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

all_tokens = list(itertools.chain.from_iterable(df['processed_text']))

# Extract unigrams, bigrams, and trigrams
unigrams = Counter(all_tokens)
bigrams = Counter(extract_ngrams(all_tokens, 2))
trigrams = Counter(extract_ngrams(all_tokens, 3))

# Display the 20 most common unigrams, bigrams, and trigrams
print("\nMost Common Unigrams:")
display(unigrams.most_common(20))

print("\nMost Common Bigrams:")
display(bigrams.most_common(20))

print("\nMost Common Trigrams:")
display(trigrams.most_common(20))


Most Common Unigrams:


[('good', 125),
 ('app', 108),
 ('nice', 37),
 ('work', 34),
 ('update', 24),
 ('cbe', 24),
 ('well', 20),
 ('bank', 20),
 ('ok', 19),
 ('transfer', 19),
 ('transaction', 18),
 ('use', 17),
 ('application', 16),
 ('service', 16),
 ('easy', 16),
 ('mobile', 15),
 ('like', 14),
 ('fast', 13),
 ('time', 13),
 ('bad', 12)]


Most Common Bigrams:


[(('good', 'app'), 25),
 (('good', 'good'), 18),
 (('nice', 'app'), 14),
 (('app', 'good'), 13),
 (('easy', 'use'), 8),
 (('ok', 'good'), 7),
 (('good', 'nice'), 7),
 (('mobile', 'banking'), 7),
 (('good', 'application'), 6),
 (('nice', 'good'), 5),
 (('update', 'app'), 5),
 (('mobile', 'app'), 5),
 (('app', 'nice'), 5),
 (('good', 'service'), 5),
 (('good', 'job'), 5),
 (('well', 'good'), 5),
 (('app', 'work'), 4),
 (('good', 'easy'), 4),
 (('transfer', 'telebirr'), 4),
 (('app', 'ok'), 3)]


Most Common Trigrams:


[(('good', 'app', 'good'), 6),
 (('good', 'good', 'good'), 5),
 (('good', 'good', 'nice'), 3),
 (('good', 'good', 'app'), 3),
 (('good', 'nice', 'app'), 3),
 (('good', 'app', 'financial'), 2),
 (('good', 'wow', 'good'), 2),
 (('wow', 'good', 'application'), 2),
 (('well', 'ok', 'good'), 2),
 (('app', 'good', 'good'), 2),
 (('internet', 'base', 'mobile'), 2),
 (('base', 'mobile', 'app'), 2),
 (('mobile', 'app', 'cbebirr'), 2),
 (('ግን', 'አንድ', 'ችግር'), 2),
 (('good', 'application', 'good'), 2),
 (('cbe', 'mobile', 'banking'), 2),
 (('good', 'service', 'good'), 2),
 (('good', 'easy', 'use'), 2),
 (('safaricom', 'network', 'app'), 2),
 (('app', 'good', 'app'), 2)]

### Grouping Keywords into Thematic Categories

Based on the extracted unigrams, bigrams, and trigrams, I will now group related keywords into 3-5 overarching themes that are relevant to customer feedback for banking apps. The goal is to identify recurring issues or areas of praise.

Here is my proposed grouping logic and the resulting themes:

1.  **Performance & Functionality:** Keywords related to app speed, crashes, bugs, errors, and general operational effectiveness.
    *   *Keywords:* `app`, `work`, `good`, `open`, `transaction`, `update`, `money`, `problem`, `bug`, `issue`, `error`, `slow`, `fast`, `crash`, `transfer`.

2.  **User Experience (UX) & Design:** Keywords related to the user interface, ease of use, navigation, and overall aesthetic.
    *   *Keywords:* `easy`, `use`, `interface`, `design`, `simple`, `user`, `friendly`, `complicated`, `layout`, `experience`, `navigation`, `smooth`.

3.  **Account Management & Access:** Keywords about login, access, account features, security, and personal information management.
    *   *Keywords:* `login`, `account`, `access`, `secure`, `security`, `otp`, `fingerprint`, `password`, `verification`, `customer`, `service`, `support`.

4.  **Features & Services:** Keywords related to specific banking features, missing functionalities, or requests for new services.
    *   *Keywords:* `feature`, `service`, `option`, `add`, `need`, `request`, `new`, `useful`, `helpful`, `card`, `payment`, `bill`.

This grouping aims to categorize feedback into actionable areas for the bank to address.

In [17]:
# Define the thematic categories and their associated keywords
theme_keywords = {
    'Performance & Functionality': [
        'app', 'work', 'good', 'open', 'transaction', 'update', 'money', 'problem', 'bug', 'issue',
        'error', 'slow', 'fast', 'crash', 'transfer', 'fail', 'fix', 'stable', 'performance', 'respond',
        'lag', 'load', 'freeze', 'broken', 'connection'
    ],
    'User Experience (UX) & Design': [
        'easy', 'use', 'interface', 'design', 'simple', 'user', 'friendly', 'complicated', 'layout',
        'experience', 'navigation', 'smooth', 'intuitive', 'look', 'ugly', 'confuse', 'hard'
    ],
    'Account Management & Access': [
        'login', 'account', 'access', 'secure', 'security', 'otp', 'fingerprint', 'password', 'verification',
        'customer', 'service', 'support', 'register', 'sign', 'authenticate', 'enroll', 'block', 'recover',
        'phone', 'number', 'email'
    ],
    'Features & Services': [
        'feature', 'service', 'option', 'add', 'need', 'request', 'new', 'useful', 'helpful', 'card',
        'payment', 'bill', 'loan', 'credit', 'debit', 'deposit', 'withdraw', 'history', 'statement', 'send',
        'receive', 'notification', 'alert', 'digital'
    ]
}

def assign_themes(processed_tokens):
    assigned = []
    for theme, keywords in theme_keywords.items():
        # Check for presence of any keyword from the theme in the processed tokens
        if any(keyword in processed_tokens for keyword in keywords):
            assigned.append(theme)
    return assigned if assigned else ['Uncategorized']

df['assigned_themes'] = df['processed_text'].apply(assign_themes)

display(df[['review_text', 'processed_text', 'assigned_themes']].head())

all_assigned_themes = list(itertools.chain.from_iterable(df['assigned_themes']))
theme_counts = Counter(all_assigned_themes)

print("\nOverall Theme Distribution:")
display(theme_counts.most_common())

,review_text,processed_text,assigned_themes
0,good apps,"[good, app]",[Performance & Functionality]
1,ok,[ok],[Uncategorized]
2,this update got crazy i don't know what's goin...,"[update, get, crazy, know, go, app, mal, funct...",[Performance & Functionality]
3,thanks for you 😘,[thank],[Uncategorized]
4,it's okay,[okay],[Uncategorized]



Overall Theme Distribution:


[('Performance & Functionality', 254),
 ('Uncategorized', 164),
 ('Features & Services', 51),
 ('Account Management & Access', 43),
 ('User Experience (UX) & Design', 35)]

### Aggregate Thematic Analysis by Bank and Star Rating

Now, let's aggregate the identified themes to see their distribution across different banks and star ratings. This will help us understand which issues or positive feedback are more prevalent for specific banks or at certain star rating levels.

In [18]:

df_themes_exploded = df.explode('assigned_themes')


themes_by_bank_rating = df_themes_exploded.groupby(['bank', 'rating', 'assigned_themes']).size().unstack(fill_value=0)

display(themes_by_bank_rating)

themes_by_bank = df_themes_exploded.groupby('bank')['assigned_themes'].value_counts().unstack(fill_value=0)

display(themes_by_bank)

assigned_themes  Account Management & Access  Features & Services  \
bank     rating                                                     
CBE Bank 1                                13                   17   
         2                                 1                    3   
         3                                 4                    3   
         4                                 2                    5   
         5                                23                   23   

assigned_themes  Performance & Functionality  Uncategorized  \
bank     rating                                               
CBE Bank 1                                43             15   
         2                                 8              2   
         3                                21              6   
         4                                19             18   
         5                               163            123   

assigned_themes  User Experience (UX) & Design  
bank     rating                                 
CBE Bank 1                                   7  
         2                                   0  
         3                                   3  
         4                                   5  
         5                                  20

assigned_themes,Account Management & Access,Features & Services,Performance & Functionality,Uncategorized,User Experience (UX) & Design
bank,,,,,
CBE Bank,43,51,254,164,35


## Thematic Analysis: Topic Modeling with LDA (Latent Dirichlet Allocation)

Given the presence of 'Uncategorized' reviews in the keyword-based thematic analysis, let's employ Latent Dirichlet Allocation (LDA) for topic modeling. LDA is a generative statistical model that allows sets of observations to be explained by unobserved groups that explain why some parts of the data are similar. It's often used to discover abstract 'topics' that occur in a collection of documents.

I will use `gensim` and `pyLDAvis` for this task. First, I need to prepare the text data (processed tokens) into a dictionary and a corpus, which are required formats for `gensim` LDA.

In [19]:
!pip install gensim pyLDAvis -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 82.5 MB/s eta 0:00:00


In [20]:
from gensim import corpora
from gensim.models.ldamodel import LdaModel

filtered_processed_text = [tokens for tokens in df['processed_text'] if tokens]


dictionary = corpora.Dictionary(filtered_processed_text)

dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=100000)

corpus = [dictionary.doc2bow(tokens) for tokens in filtered_processed_text]

print("Number of unique tokens: %d" % len(dictionary))
print("Number of documents (reviews): %d" % len(corpus))

print("\nFirst 10 words in dictionary:")
print({id: dictionary[id] for id in list(dictionary.keys())[:10]})

print("\nFirst document in corpus (BoW format):")
print(corpus[0])

Number of unique tokens: 49
Number of documents (reviews): 445

First 10 words in dictionary:
{0: 'app', 1: 'good', 2: 'ok', 3: 'functional', 4: 'like', 5: 'update', 6: 'thank', 7: 'money', 8: 'transfer', 9: 'work'}

First document in corpus (BoW format):
[(0, 1), (1, 1)]


### Run LDA Model

Now, I'll train the LDA model. I'll start with a reasonable number of topics (e.g., 5) and we can adjust this later if needed. The model will identify sets of words that frequently occur together, which represent the underlying topics.

In [21]:

RANDOM_STATE = 42

# Build LDA model
num_topics = 5 # Can be adjusted
lda_model = LdaModel(corpus=corpus,
                       id2word=dictionary,
                       num_topics=num_topics,
                       random_state=RANDOM_STATE,
                       update_every=1,
                       chunksize=100,
                       passes=10,
                       alpha='auto',
                       per_word_topics=True)

# Print the keywords for each topic
print("\nLDA Topics and their top keywords:")
for idx, topic in lda_model.print_topics(num_words=10): # num_words can be adjusted
    print(f"Topic {idx}: {topic}")


LDA Topics and their top keywords:
Topic 0: 0.264*"nice" + 0.127*"application" + 0.119*"customer" + 0.070*"support" + 0.068*"like" + 0.057*"unable" + 0.043*"payment" + 0.040*"experience" + 0.037*"crash" + 0.034*"access"
Topic 1: 0.192*"transaction" + 0.160*"easy" + 0.139*"use" + 0.086*"bad" + 0.083*"new" + 0.064*"history" + 0.063*"thank" + 0.062*"great" + 0.032*"functional" + 0.029*"problem"
Topic 2: 0.561*"good" + 0.105*"ok" + 0.098*"transfer" + 0.081*"fix" + 0.037*"properly" + 0.023*"issue" + 0.022*"በጣም" + 0.021*"ነው" + 0.010*"problem" + 0.001*"job"
Topic 3: 0.336*"work" + 0.176*"update" + 0.117*"well" + 0.097*"open" + 0.085*"banking" + 0.068*"wow" + 0.028*"pls" + 0.017*"secure" + 0.011*"mobile" + 0.002*"customer"
Topic 4: 0.407*"app" + 0.106*"cbe" + 0.097*"bank" + 0.071*"time" + 0.057*"mobile" + 0.053*"fast" + 0.052*"service" + 0.042*"need" + 0.037*"ethiopia" + 0.031*"excellent"


### Visualize LDA Topics

To better understand the identified topics and their inter-relationships, I will use `pyLDAvis` for interactive visualization. This tool helps in interpreting the topics by showing prominent terms for each topic and the distances between topics.

In [22]:
import pyLDAvis.gensim_models as gensim_models
import pyLDAvis

# Prepare data for visualization
lda_display = gensim_models.prepare(lda_model, corpus, dictionary, sort_topics=False)

pyLDAvis.display(lda_display)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Assign Dominant Topic to Each Review

Now that we have the LDA topics, I'll assign the dominant topic (the topic with the highest probability) to each original review. This will allow us to see the distribution of these newly discovered topics across the reviews, and potentially understand the content of the 'Uncategorized' reviews from the previous keyword-based analysis.

In [23]:
def get_dominant_topic(ldamodel, corpus_item):
    # Get topic distributions for the document
    topic_probs = ldamodel.get_document_topics(corpus_item, minimum_probability=0)
    # Sort by probability in descending order
    sorted_topic_probs = sorted(topic_probs, key=lambda x: x[1], reverse=True)
    # Return the topic ID and its probability for the dominant topic
    if sorted_topic_probs:
        return sorted_topic_probs[0][0], sorted_topic_probs[0][1]
    return None, 0.0

# Create a list to store dominant topics for each review
dominant_topics = []
# Keep track of indices for original df, as some reviews might have been filtered out if their processed_text was empty
original_df_indices = df[df['processed_text'].apply(len) > 0].index.tolist()

for i, corpus_item in enumerate(corpus):
    topic_id, prob = get_dominant_topic(lda_model, corpus_item)
    dominant_topics.append({'review_index': original_df_indices[i], 'dominant_topic_id': topic_id, 'dominant_topic_prob': prob})

# Create a DataFrame from dominant topics
df_dominant_topics = pd.DataFrame(dominant_topics).set_index('review_index')

# Merge dominant topics back to the original DataFrame
df = df.join(df_dominant_topics)

# Fill NaN values for reviews that had empty processed_text
df['dominant_topic_id'] = df['dominant_topic_id'].fillna(-1).astype(int) # Use -1 for reviews without a topic
df['dominant_topic_prob'] = df['dominant_topic_prob'].fillna(0.0)

# Display reviews with their dominant topics
display(df[['review_text', 'processed_text', 'assigned_themes', 'dominant_topic_id', 'dominant_topic_prob']].head())

print("\nDistribution of Dominant Topics:")
display(df['dominant_topic_id'].value_counts().sort_index())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,review_text,processed_text,assigned_themes,dominant_topic_id,dominant_topic_prob
0,good apps,"[good, app]",[Performance & Functionality],4,0.400611
1,ok,[ok],[Uncategorized],2,0.527508
2,this update got crazy i don't know what's goin...,"[update, get, crazy, know, go, app, mal, funct...",[Performance & Functionality],0,0.379165
3,thanks for you 😘,[thank],[Uncategorized],1,0.400634
4,it's okay,[okay],[Uncategorized],4,0.303745



Distribution of Dominant Topics:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
dominant_topic_id,
-1,5
0,41
1,31
2,126
3,46
4,201


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Aggregating LDA Topics by Bank and Star Rating

Finally, let's aggregate the dominant LDA topics to see their distribution across banks and star ratings, similar to how we analyzed the keyword-based themes.

In [24]:
# Aggregate dominant topic counts by bank and star rating
lda_topics_by_bank_rating = df.groupby(['bank', 'rating', 'dominant_topic_id']).size().unstack(fill_value=0)

display(lda_topics_by_bank_rating)

# Aggregate dominant topic counts by bank only
lda_topics_by_bank = df.groupby('bank')['dominant_topic_id'].value_counts().unstack(fill_value=0)

display(lda_topics_by_bank)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

dominant_topic_id  -1   0   1   2   3    4
bank     rating                           
CBE Bank 1          0  11   6  12  11   25
         2          0   1   1   4   0    4
         3          0   0   3   9   3   15
         4          0   3   6   8   6   18
         5          5  26  15  93  26  139

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

dominant_topic_id,-1,0,1,2,3,4
bank,,,,,,
CBE Bank,5,41,31,126,46,201


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Comparing Keyword-based Themes with LDA Topics

After reviewing the LDA visualization and topic keywords, we can compare these automatically discovered topics with our manually defined themes. This will provide a more comprehensive thematic understanding of the reviews.